# Notebook 01 — Dataset Exploration and Preprocessing
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** What does our dataset look like, and are our DataLoaders correctly configured before we invest GPU time in training?

**Phase 1 rubric targets:**
- Visualize >=5 sample images with labels (1 pt)
- Class distribution bar chart (1 pt)
- Functional train / val / test DataLoaders (1 pt)

In [ ]:
# ── Cell 0A: Environment setup (Drive + GitHub) ────────────
# Why: mounts Drive for artifact persistence, clones or pulls
# the repo from GitHub to guarantee latest src/ modules,
# and installs plotnine. Run first on every Colab session.
import os
import subprocess
import sys

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/landmark-classifier'):
        # First session: clone the full repo
        subprocess.run([
            'git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'
        ], check=True)
    else:
        # Subsequent sessions: pull latest commits (src/ updates)
        # Why: without pull, stale modules run even after GitHub commits
        subprocess.run([
            'git', '-C', '/content/landmark-classifier',
            'pull', 'origin', 'master'
        ], check=True)
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    sys.path.insert(0, '/content/landmark-classifier')
    print('Colab environment ready')
else:
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
import logging
from src.config import (
    DEVICE, EXPERIMENTS_DIR, NUM_CLASSES, SEED, TEST_DIR, TRAIN_DIR,
)
from src.utils import set_seed

logging.basicConfig(level=logging.INFO)
set_seed(SEED)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device          : {DEVICE}')
print(f'Train dir       : {TRAIN_DIR}  exists={TRAIN_DIR.exists()}')
print(f'Test dir        : {TEST_DIR}   exists={TEST_DIR.exists()}')
print(f'Num classes     : {NUM_CLASSES}')
print(f'EXPERIMENTS_DIR : {EXPERIMENTS_DIR}')


## 1. Dataset Structure Audit

In [ ]:
# ── Cell 2: Dataset structure audit ────────────────────────
import os

train_classes = sorted(os.listdir(TRAIN_DIR))
test_classes  = sorted(os.listdir(TEST_DIR))
print(f'Train classes : {len(train_classes)}')
print(f'Test  classes : {len(test_classes)}')
print(f'Classes match : {train_classes == test_classes}')
print(f'First 5       : {train_classes[:5]}')
train_counts = {
    cls: len(list((TRAIN_DIR / cls).glob('*')))
    for cls in train_classes
}
total_train = sum(train_counts.values())
print(f'Total train   : {total_train}')
print(f'Avg per class : {total_train / len(train_classes):.1f}')


## 2. Class Distribution — BI Visualization

In [ ]:
# ── Cell 3: BI class distribution plot ─────────────────────
# UCB palette: Gold = majority classes (>1.5x mean) — bias source
import matplotlib.pyplot as plt
import numpy as np

UCB_BLUE      = '#003262'
UCB_GOLD      = '#FDB515'
UCB_DARK_GOLD = '#C4820E'

names      = list(train_counts.keys())
counts     = list(train_counts.values())
mean_count = np.mean(counts)
colors     = [UCB_GOLD if c > mean_count * 1.5 else UCB_BLUE for c in counts]

fig, ax = plt.subplots(figsize=(22, 5), facecolor='white')
ax.bar(range(len(names)), counts, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(mean_count, color=UCB_DARK_GOLD, linestyle='--', linewidth=1.2,
           label=f'Mean: {mean_count:.0f} images/class')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('_', ' ') for n in names], rotation=90, fontsize=6)
ax.set_title('Landmark Dataset — Class Distribution\n'
             'Gold = majority classes (>1.5x mean)',
             fontsize=12, fontweight='bold', color=UCB_BLUE)
ax.set_xlabel('Landmark Class', fontsize=9, color=UCB_BLUE)
ax.set_ylabel('Number of Images', fontsize=9, color=UCB_BLUE)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
out = EXPERIMENTS_DIR / 'class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')


## 3. Sample Images with Labels

In [ ]:
# ── Cell 4: Sample images ───────────────────────────────────
import random
from torchvision.datasets import ImageFolder

raw_ds     = ImageFolder(str(TRAIN_DIR))
sample_idx = random.sample(range(len(raw_ds)), 8)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), facecolor='white')
for ax, idx in zip(axes.flatten(), sample_idx):
    img, label = raw_ds[idx]
    ax.imshow(img)
    ax.set_title(raw_ds.classes[label].replace('_', ' '),
                 fontsize=8, color=UCB_BLUE, fontweight='bold')
    ax.axis('off')
fig.suptitle('Sample Landmark Images with Ground Truth Labels',
             fontsize=12, fontweight='bold', color=UCB_BLUE)
plt.tight_layout()
out = EXPERIMENTS_DIR / 'sample_images.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')


## 4. DataLoaders — Build and Verify

In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────
# Why rebuild every session: Colab runtime resets lose in-memory
# objects. Rebuilding is fast and guarantees a clean state.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)


## 5. Augmentation Preview

In [ ]:
# ── Cell 6: Augmentation preview ───────────────────────────
import torch
from PIL import Image
from src.data import get_transforms

sample_class = class_names[0]
sample_path  = next((TRAIN_DIR / sample_class).glob('*'))
raw_img      = Image.open(sample_path).convert('RGB')
transform_aug = get_transforms(augment=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='white')
axes[0].imshow(raw_img)
axes[0].set_title('Original', color=UCB_BLUE, fontweight='bold')
for i, ax in enumerate(axes[1:], 1):
    aug_tensor = transform_aug(raw_img)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = (aug_tensor * std + mean).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(img_display)
    ax.set_title(f'Augmented #{i}', color=UCB_DARK_GOLD, fontweight='bold')
for ax in axes:
    ax.axis('off')
fig.suptitle(f'Augmentation Preview — {sample_class.replace("_", " ")}',
             fontsize=11, fontweight='bold', color=UCB_BLUE)
plt.tight_layout()
out = EXPERIMENTS_DIR / 'augmentation_preview.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out}')


## Phase 1 — Checklist

| Check | Status |
|---|---|
| 50 classes detected | ✅ |
| Class distribution plotted + saved to Drive | ✅ |
| >=5 sample images displayed + saved to Drive | ✅ |
| DataLoaders: batch shape [B, 3, 224, 224] | ✅ |
| Augmentation preview saved to Drive | ✅ |

**Next step:** `02_cnn_from_scratch.ipynb` — Phase 2 on Colab A100.